In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import DiskImage, DiskBooleanMask, show, mkdir
import json
from tqdm import tqdm

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_7 import ImageReviewWidget

INAT = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/flowers/inat/")
BBOXES = INAT / "bboxes"
SKIP = INAT / "skipped"
SELECTED = INAT / "selected"
PHOTOS = INAT / "photos"

i tried doing the cropping thing. its beautifully painful.  
We'll just randomise. we'll take random crops from the image. and be good. We just generate a lot more data then to make sure the randomisation works.  


In [ ]:
images = list(PHOTOS.glob("*.jpg"))
len(images)

# Check out data

Try finding bounding boxes using standard opencv heuristics.  

In [ ]:
from mtrain.neg_mask.flower import detect_flower_bboxes

In [ ]:
flower_colors = {
    # Red wraps around the 180-degree mark, so it needs two ranges
    "red_lower": {"l": np.array([0, 100, 80]), "u": np.array([10, 255, 255])},
    "red_upper": {
        "l": np.array([160, 100, 80]),
        "u": np.array([179, 255, 255]),
    },
    "yellow": {"l": np.array([22, 100, 100]), "u": np.array([35, 255, 255])},
    "orange": {"l": np.array([11, 100, 100]), "u": np.array([21, 255, 255])},
    "green": {"l": np.array([36, 60, 60]), "u": np.array([89, 255, 255])},
    "blue_purple": {
        "l": np.array([90, 80, 80]),
        "u": np.array([155, 255, 255]),
    },
    "pink_bright": {
        "l": np.array([140, 80, 100]),
        "u": np.array([165, 255, 255]),
    },
    "white": {"l": np.array([0, 0, 200]), "u": np.array([180, 40, 255])},
    # Gray/Silver/Rocks: Low saturation across all hues
    "gray_stones": {"l": np.array([0, 0, 40]), "u": np.array([180, 50, 180])},
    # Brown/Mud/Mulch: Low-to-mid saturation, low-to-mid brightness
    # (Essentially dark, desaturated oranges/reds)
    "brown_dirt": {"l": np.array([0, 20, 20]), "u": np.array([30, 100, 120])},
    # Deep Shadows: Very low brightness regardless of color
    "shadows": {"l": np.array([0, 0, 0]), "u": np.array([180, 255, 40])},
    # Bright Concrete/Sun-bleached wood: High brightness, very low saturation
    "bright_neutrals": {"l": np.array([0, 0, 180]), "u": np.array([180, 30, 255])},
}


def get_guess_masks(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return []

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Convert BGR to HSV for better color filtering
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    # Initialize masks dictionary for visualization

    masks = {}
    masks["image"] = image_rgb

    for color_name, rg in flower_colors.items():
        low, high = rg["l"], rg["u"]
        masks[color_name] = cv2.inRange(hsv, low, high)
    return masks

    lower_green = np.array([40, 50, 50])
    upper_green = np.array([80, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    masks["green"] = green_mask

    # Brown exclusion (ground, bark, soil, wooden surfaces)
    # Brown colors typically have hue around 10-25 degrees, medium-low saturation, medium-low brightness
    lower_brown1 = np.array([8, 30, 30])  # Light brown/tan
    upper_brown1 = np.array([25, 200, 150])
    brown_mask1 = cv2.inRange(hsv, lower_brown1, upper_brown1)

    # Dark brown (bark, dark soil)
    lower_brown2 = np.array([5, 50, 20])  # Dark brown
    upper_brown2 = np.array([30, 255, 100])
    brown_mask2 = cv2.inRange(hsv, lower_brown2, upper_brown2)

    # Combine brown masks
    brown_mask = cv2.bitwise_or(brown_mask1, brown_mask2)
    masks["brown"] = brown_mask

    return masks


flower_keys = [
    "red_lower",
    "red_upper",
    "yellow",
    "orange",
    "green",
    "blue_purple",
    "pink_bright",
    "white",
]
bg_keys = ["gray_stones", "brown_dirt", "shadows", "bright_neutrals"]


def combine_masks(masks):
    flower_mask = np.zeros(masks["red_lower"].shape, dtype=bool)
    for k in flower_keys:
        if k == "blue_purple":
            continue
        flower_mask |= masks[k] == 255
    bg_mask = np.zeros(masks["red_lower"].shape, dtype=bool)
    for k in bg_keys:
        bg_mask |= masks[k] == 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    bg_mask = bg_mask.astype(np.uint8)
    bg_mask = cv2.morphologyEx(bg_mask, cv2.MORPH_OPEN, kernel)
    bg_mask = cv2.morphologyEx(bg_mask, cv2.MORPH_CLOSE, kernel)

    flower_mask = flower_mask.astype(np.uint8)
    flower_mask = cv2.morphologyEx(flower_mask, cv2.MORPH_OPEN, kernel)
    flower_mask = cv2.morphologyEx(flower_mask, cv2.MORPH_CLOSE, kernel)

    return flower_mask.astype(bool), bg_mask.astype(bool)

In [ ]:
idx = 14
masks = get_guess_masks(images[idx])
flower_mask, bg_mask = combine_masks(masks)
resized = cv2.resize(
    # cv2.GaussianBlur(masks["image"], (3,3), 3),
    masks["image"],
    (130, 130),
    interpolation=cv2.INTER_AREA,
)
# mask_rsz = cv2.resize(flower_mask.astype(np.uint8), (130,130), interpolation=cv2.INTER_NEAREST).astype(bool)
# blurred = cv2.GaussianBlur(resized, (3, 3), 1)
show([masks["image"], resized], (20, 20), ncols=2, axis="on")

# Only keep far ish images

This seems to be good enough i would say, the mask. for finding flowers.  
Now the other problem is that it looks like the images are quite close. We need to blur them first.  ()

In a simple resize, faraway objects work better

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_8 import ImageReviewWidget

In [ ]:
INAT = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/flowers/inat/")
BBOXES = INAT / "bboxes"
SKIP = INAT / "skipped"
SELECTED = INAT / "selected"
PHOTOS = INAT / "photos"

widget = ImageReviewWidget(PHOTOS, SELECTED, SKIP)

In [ ]:
widget.ui()

# Model training

In [ ]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
ROCKS = NEG_MASKING_V1 / "rocks"
CLASSI = ROCKS / "classification"
FLOWERS = CLASSI / "flowers"
CROP_LEVEL = CLASSI / "crop_level"

def get_flower_images_in_classification_dir():
    added_images = set()
    added_crops = {}
    for d in ["trash", "other"]:
        label_direc = CROP_LEVEL / d
        for d in label_direc.glob("*"):
            if not d.is_dir():
                continue
            splits = d.name.split("_")
            actual_name = "_".join(splits[:-1])
            added_images.add(actual_name)
            added_crops[d] = actual_name

    mapi_flower_images = [i.resolve() for i in FLOWERS.rglob("image.jpg")]
    mapi_flower_images = {i.parent.name: i for i in mapi_flower_images if i.parent.name in added_images}

    to_check = []
    for crop, image in added_crops.items():
        if image in mapi_flower_images:
            to_check.append((crop, mapi_flower_images[image]))
    return to_check


In [ ]:
# create ds first
DS_SIZE = 100
CROP_SIZE = 60
BASE_DIR = mkdir(INAT / "train" / "size-100_crop-60")
IMAGES_DIR = mkdir(BASE_DIR / "images")
POSITIVES = mkdir(IMAGES_DIR / "positive")
TRASH_DIR = mkdir(IMAGES_DIR / "trash")
MAPILLARY_SAMPLES_DIR = mkdir(IMAGES_DIR / "mapillary_random")
# DEST = mkdir(INAT / "train" / "size_130" / "images")

def put_photos_to_pos_dir(photos, pos_dir, ds_size):
    all_photos = list(photos.glob("*"))
    for path in tqdm(all_photos):
        image = cv2.imread(path)
        image = cv2.resize(image, (ds_size,ds_size), interpolation=cv2.INTER_AREA)
        cv2.imwrite(pos_dir / path.name, image)

In [ ]:
put_photos_to_pos_dir(PHOTOS, POSITIVES, DS_SIZE)

In [ ]:
# now get images frmo mapillary of the crop size
# what should i take from mpaillary images?

# i would not want elevated vegetation, it might skew the data (im anyways going for "green") lol
# i would definitely want the trash categories, getting some from the trash dir is useful
# we are basically detecting: if flowers or leaves only, keep them. if anything else in them, skip
# in this case, having mapillary trash data as negatives is useful
# the other case is using random photos from already existing mapillary samples (i could use delhi)
# in this randomization, we skip vegetation altogether
# /Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test: has 2000 sample images
# /Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/trash -> 2600 images

# flowers + greens: 3248
# now we need some 2x negatives
# ive got trash, which is a strong negative, 2600 images
# delhi has 2000 images, we simply take random crop from each if it is not vegetation

## check out neg samples

In [ ]:
from mtrain.neg_mask.flower.neg_ds import get_crop_level_trash_from_dirs
from fastai.basics import get_image_files

CROP_LEVEL_TRASH_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/trash")

CROP_LEVEL_OTHER_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/other")

classi_flower_crop_and_img = get_flower_images_in_classification_dir()
crops_to_see = set([f[0].resolve() for f in classi_flower_crop_and_img])

dirs = [
    c
    for c in (list(CROP_LEVEL_TRASH_DIR.glob("*")) + list(CROP_LEVEL_OTHER_DIR.glob("*")))
    # if c in crops_to_see
]


it = get_crop_level_trash_from_dirs(dirs, CROP_SIZE)
files = get_image_files(POSITIVES)
flower_it = iter(files)

In [ ]:
show([next(it), plt.imread(next(flower_it))]

## put trash neg samples

In [ ]:
from mtrain.neg_mask.flower.neg_ds import get_crop_level_trash_from_dirs
from fastai.basics import get_image_files

CROP_LEVEL_TRASH_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level/trash")

dirs = [
    c.resolve() for c in CROP_LEVEL_TRASH_DIR.glob("*")
]

for d, crop in tqdm(get_crop_level_trash_from_dirs(dirs, CROP_SIZE), total=len(dirs)):
    dest = TRASH_DIR / f"{d.name}.jpg"
    # print(dest)
    plt.imsave(dest, crop)

## put mapillary random samples

In [ ]:
def get_random_crop(image, crop_height, crop_width):
    h, w = image.shape[:2]
    if crop_height > h or crop_width > w:
        raise ValueError("Crop size must be smaller than image size")
    max_x = w - crop_width
    max_y = h - crop_height
    x = np.random.randint(0, max_x + 1)
    y = np.random.randint(0, max_y + 1)
    cropped_image = image[y:y + crop_height, x:x + crop_width]
    return cropped_image.copy()

In [ ]:
def get_random_delhi_crops():
    RANDOM_DELHI = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test")


    dirs = list(RANDOM_DELHI.glob("*"))
    for path in dirs:
        if not (path / "image.jpg").exists():
            continue
        image = plt.imread(path / "image.jpg")
        image = cv2.resize(image, (1024, 1024))
        # random crop
        crop = get_random_crop(image, CROP_SIZE*2, CROP_SIZE*2)
        yield crop

In [ ]:

for i, crop in enumerate(get_random_delhi_crops()):
    plt.imsave(MAPILLARY_SAMPLES_DIR / f"{i}.jpg", crop)

In [ ]:
plt.imshow(next(it))

## prepare ds

In [ ]:
DS_ROOT = mkdir(BASE_DIR / "dataset")
positives = mkdir(DS_ROOT / "pos")
negatives = mkdir(DS_ROOT / "negs")

In [ ]:
flower_paths = list(POSITIVES.glob("*.jpg"))
trash_paths = list(TRASH_DIR.glob("*.jpg"))
mapillary_paths = list(MAPILLARY_SAMPLES_DIR.glob("*.jpg"))

In [ ]:
len(flower_paths), len(trash_paths), len(mapillary_paths)

In [ ]:
import shutil
for f in flower_paths:
    shutil.copy(f, positives / f.name)

In [ ]:
from uuid import uuid4
for i, f in enumerate(trash_paths):
    shutil.copy(f, negatives / f"trash_{i}.jpg")

In [ ]:
from uuid import uuid4
for i, f in enumerate(mapillary_paths):
    shutil.copy(f, negatives / f"mapi_{i}.jpg")

# Train model

In [ ]:
LOG_ROOT = "/Users/hariomnarang/Desktop/personal/roads/datasets/flowers/inat/train/size-100_crop-60/models"

In [ ]:
from fastai.vision.all import *


def label_func(x):
    if x.startswith("mapi") or x.startswith("trash"):
        return "neg"
    else:
        return "pos"

dls = ImageDataLoaders.from_name_func(
    LOG_ROOT, get_image_files(DS_ROOT), valid_pct=0.2, seed=42,
    label_func=label_func, item_tfms=RandomResizedCrop(CROP_SIZE), batch_tfms=aug_transforms())

# learn = vision_learner(dls, resnet34, metrics=error_rate)
# learn.fine_tune(1)

In [ ]:
dls.show_batch()

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn = learn.remove_cb(ProgressCallback)
learn.fine_tune(1)

In [ ]:
learn.fit_one_cycle(15)

In [ ]:
learn.export("wuth-aug-tfms-iter-15.pkl")

# Check crop level existing

In [ ]:
to_check

In [ ]:
it = iter(to_check)


In [ ]:
crop

In [ ]:
crop, img = next(it)
show([plt.imread(crop / "image.jpg"), plt.imread(img)])

In [ ]:
to_check

In [ ]:
show([masks["image"], flower_mask, bg_mask], ncols=3, cmap="gray")

In [ ]:
masks["red_lower"].shape

In [ ]:
masks.keys()

A lot of approaches for finding bounding boxes are not working at all lol.  

In [ ]:
def get_hue_ranges():
    pass

In [ ]:
def get_masks(image_path):
    pass

In [ ]:
def detect_non_green_flowers(
    image_path, min_area=100, max_area_ratio=0.8, return_masks=False
):
    """
    Detect flowers using color-based heuristic: flowers are typically non-green and non-brown.
    Includes support for white flowers and other flower colors.

    Args:
        image_path: Path to image file
        min_area: Minimum contour area to consider
        max_area_ratio: Maximum area ratio relative to image (to filter out background)
        return_masks: If True, return intermediate masks for visualization

    Returns:
        If return_masks=False: List of bounding box dictionaries with x1, y1, x2, y2
        If return_masks=True: (bboxes, masks_dict, image_rgb) where masks_dict contains all intermediate masks
    """
    # Load image
    image = cv2.imread(str(image_path))
    if image is None:
        if return_masks:
            return [], {}, None
        return []

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Convert BGR to HSV for better color filtering
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)

    import numpy as np

    # Initialize masks dictionary for visualization
    masks = {}

    # Store original image and HSV channels
    masks["original_image"] = image_rgb
    masks["hue_channel"] = h
    masks["saturation_channel"] = s
    masks["value_channel"] = v

    # Define color ranges to exclude (green and brown)
    # Green exclusion (foliage)
    lower_green = np.array([40, 50, 50])
    upper_green = np.array([80, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    masks["green_exclusion"] = green_mask

    # Brown exclusion (ground, bark, soil, wooden surfaces)
    # Brown colors typically have hue around 10-25 degrees, medium-low saturation, medium-low brightness
    lower_brown1 = np.array([8, 30, 30])  # Light brown/tan
    upper_brown1 = np.array([25, 200, 150])
    brown_mask1 = cv2.inRange(hsv, lower_brown1, upper_brown1)

    # Dark brown (bark, dark soil)
    lower_brown2 = np.array([5, 50, 20])  # Dark brown
    upper_brown2 = np.array([30, 255, 100])
    brown_mask2 = cv2.inRange(hsv, lower_brown2, upper_brown2)

    # Combine brown masks
    brown_mask = cv2.bitwise_or(brown_mask1, brown_mask2)
    masks["brown_exclusion"] = brown_mask
    masks["light_brown"] = brown_mask1
    masks["dark_brown"] = brown_mask2

    # Combine exclusion masks (green + brown)
    exclusion_mask = cv2.bitwise_or(green_mask, brown_mask)
    masks["combined_exclusion"] = exclusion_mask

    # Create multiple flower detection masks
    flower_masks = []

    # 1. Colorful flowers (high saturation, non-green, non-brown)
    colorful_mask = cv2.threshold(s, 80, 255, cv2.THRESH_BINARY)[1]
    colorful_flowers = cv2.bitwise_and(colorful_mask, cv2.bitwise_not(exclusion_mask))
    flower_masks.append(colorful_flowers)
    masks["colorful_flowers"] = colorful_flowers

    # 2. White flowers (high value/brightness, low saturation, non-green, non-brown)
    # White flowers have high brightness but low saturation
    bright_mask = cv2.threshold(v, 180, 255, cv2.THRESH_BINARY)[1]  # Very bright
    low_saturation_mask = cv2.threshold(s, 60, 255, cv2.THRESH_BINARY_INV)[
        1
    ]  # Low saturation
    white_flower_mask = cv2.bitwise_and(bright_mask, low_saturation_mask)
    # Remove green and brown areas
    white_flower_mask = cv2.bitwise_and(
        white_flower_mask, cv2.bitwise_not(exclusion_mask)
    )
    flower_masks.append(white_flower_mask)
    masks["white_flowers"] = white_flower_mask
    masks["bright_regions"] = bright_mask
    masks["low_saturation_regions"] = low_saturation_mask

    # 3. Pale/pastel flowers (medium brightness, low-medium saturation, non-green, non-brown)
    medium_bright_mask = cv2.threshold(v, 120, 255, cv2.THRESH_BINARY)[1]
    medium_saturation_mask = cv2.inRange(s, 30, 120)  # Medium saturation range
    pale_flower_mask = cv2.bitwise_and(medium_bright_mask, medium_saturation_mask)
    pale_flower_mask = cv2.bitwise_and(
        pale_flower_mask, cv2.bitwise_not(exclusion_mask)
    )
    flower_masks.append(pale_flower_mask)
    masks["pale_flowers"] = pale_flower_mask
    masks["medium_bright_regions"] = medium_bright_mask
    masks["medium_saturation_regions"] = medium_saturation_mask

    # 4. Dark flowers (lower brightness but still colorful, non-green, non-brown)
    dark_bright_mask = cv2.inRange(v, 80, 180)  # Medium brightness
    dark_colorful_mask = cv2.threshold(s, 100, 255, cv2.THRESH_BINARY)[
        1
    ]  # High saturation
    dark_flower_mask = cv2.bitwise_and(dark_bright_mask, dark_colorful_mask)
    dark_flower_mask = cv2.bitwise_and(
        dark_flower_mask, cv2.bitwise_not(exclusion_mask)
    )
    flower_masks.append(dark_flower_mask)
    masks["dark_flowers"] = dark_flower_mask

    # 5. Yellow flowers (special handling - might be confused with browns)
    # Yellow flowers typically have hue 20-35, high saturation, high brightness
    yellow_hue_mask = cv2.inRange(h, 20, 35)  # Yellow hue range
    yellow_saturation_mask = cv2.threshold(s, 120, 255, cv2.THRESH_BINARY)[
        1
    ]  # High saturation
    yellow_brightness_mask = cv2.threshold(v, 150, 255, cv2.THRESH_BINARY)[
        1
    ]  # High brightness
    yellow_flower_mask = cv2.bitwise_and(yellow_hue_mask, yellow_saturation_mask)
    yellow_flower_mask = cv2.bitwise_and(yellow_flower_mask, yellow_brightness_mask)
    # Don't exclude yellow flowers even if they might overlap with brown range
    flower_masks.append(yellow_flower_mask)
    masks["yellow_flowers"] = yellow_flower_mask
    masks["yellow_hue_regions"] = yellow_hue_mask

    # Combine all flower masks
    combined_mask = np.zeros_like(green_mask)
    for mask in flower_masks:
        combined_mask = cv2.bitwise_or(combined_mask, mask)
    masks["combined_flower_mask_raw"] = combined_mask.copy()

    # Apply morphological operations to clean up the mask
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_OPEN, kernel)
    combined_mask = cv2.morphologyEx(combined_mask, cv2.MORPH_CLOSE, kernel)
    masks["combined_flower_mask_cleaned"] = combined_mask

    # Additional filtering - remove very large regions (likely background)
    # Find contours and filter out ones that are too large to be flowers
    temp_contours, _ = cv2.findContours(
        combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    height, width = image.shape[:2]
    max_single_area = width * height * 0.3  # No single flower should be >30% of image

    # Create refined mask excluding overly large regions
    refined_mask = np.zeros_like(combined_mask)
    for contour in temp_contours:
        area = cv2.contourArea(contour)
        if area < max_single_area:  # Keep reasonably sized regions
            cv2.fillPoly(refined_mask, [contour], 255)
    masks["final_mask"] = refined_mask

    # Find final contours
    contours, _ = cv2.findContours(
        refined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    # Get image dimensions for area filtering
    max_area = width * height * max_area_ratio

    # Extract bounding boxes from contours
    bboxes = []
    for contour in contours:
        area = cv2.contourArea(contour)

        # Filter by area
        if min_area <= area <= max_area:
            x, y, w, h = cv2.boundingRect(contour)

            # Filter out very thin or very wide rectangles (likely not flowers)
            aspect_ratio = w / h if h > 0 else 0
            if 0.3 <= aspect_ratio <= 3.0:  # Reasonable aspect ratios for flowers
                # Calculate confidence based on multiple factors
                size_score = min(1.0, area / 10000)

                # Analyze the region to determine flower type
                roi_hsv = hsv[y : y + h, x : x + w]
                roi_h = roi_hsv[:, :, 0]
                roi_s = roi_hsv[:, :, 1]
                roi_v = roi_hsv[:, :, 2]

                avg_hue = np.mean(roi_h)
                avg_brightness = np.mean(roi_v)
                avg_saturation = np.mean(roi_s)

                # Determine flower type and confidence bonuses
                flower_type = "other"
                type_bonus = 0.0

                # White flower detection
                if avg_brightness > 180 and avg_saturation < 60:
                    flower_type = "white"
                    type_bonus = 0.3  # Bonus for white flowers

                # Yellow flower detection (special handling)
                elif (
                    20 <= avg_hue <= 35
                    and avg_saturation > 120
                    and avg_brightness > 150
                ):
                    flower_type = "yellow"
                    type_bonus = 0.25  # Bonus for yellow flowers

                # Colorful flower detection
                elif avg_saturation > 80:
                    flower_type = "colorful"
                    type_bonus = 0.2  # Bonus for colorful flowers

                # Pale flower detection
                elif avg_brightness > 120 and 30 <= avg_saturation <= 120:
                    flower_type = "pale"
                    type_bonus = 0.15  # Bonus for pale flowers

                confidence = min(1.0, size_score + type_bonus)

                bboxes.append(
                    {
                        "x1": float(x),
                        "y1": float(y),
                        "x2": float(x + w),
                        "y2": float(y + h),
                        "area": float(area),
                        "confidence": confidence,
                        "avg_hue": float(avg_hue),
                        "avg_brightness": float(avg_brightness),
                        "avg_saturation": float(avg_saturation),
                        "flower_type": flower_type,
                    }
                )

    # Sort by confidence and return top candidates
    bboxes.sort(key=lambda x: x["confidence"], reverse=True)
    final_bboxes = bboxes[:3]  # Return top 3 candidates maximum

    # Add final result image with bboxes to masks
    result_image = image_rgb.copy()
    colors = {
        "white": (255, 255, 255),
        "yellow": (255, 255, 0),
        "colorful": (255, 0, 255),
        "pale": (255, 192, 203),
        "other": (255, 0, 0),
    }
    for i, bbox in enumerate(final_bboxes):
        x1, y1, x2, y2 = (
            int(bbox["x1"]),
            int(bbox["y1"]),
            int(bbox["x2"]),
            int(bbox["y2"]),
        )
        flower_type = bbox.get("flower_type", "other")
        color = colors.get(flower_type, (255, 0, 0))

        cv2.rectangle(result_image, (x1, y1), (x2, y2), color, 3)
        label = f"{flower_type.title()} {i + 1}"
        cv2.putText(
            result_image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2
        )

    masks["final_result"] = result_image
    masks["detection_info"] = {
        "num_detections": len(final_bboxes),
        "image_size": (width, height),
        "parameters": {"min_area": min_area, "max_area_ratio": max_area_ratio},
    }

    if return_masks:
        return final_bboxes, masks, image_rgb
    else:
        return final_bboxes

In [ ]:
def visualize_flower_detection(image_path, show_detailed=True):
    """
    Visualize flower detection using the masks returned by the main function.

    Args:
        image_path: Path to image file
        show_detailed: If True, show detailed step-by-step visualization
    """
    # Get detection results with all masks
    bboxes, masks, image_rgb = detect_non_green_flowers(image_path, return_masks=True)

    if show_detailed:
        # Create comprehensive visualization grid
        fig, axes = plt.subplots(4, 4, figsize=(20, 20))

        # Row 1: Original, HSV channels, Final result
        axes[0, 0].imshow(masks["original_image"])
        axes[0, 0].set_title("Original Image")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(masks["hue_channel"], cmap="hsv")
        axes[0, 1].set_title("Hue Channel")
        axes[0, 1].axis("off")

        axes[0, 2].imshow(masks["saturation_channel"], cmap="gray")
        axes[0, 2].set_title("Saturation Channel")
        axes[0, 2].axis("off")

        axes[0, 3].imshow(masks["value_channel"], cmap="gray")
        axes[0, 3].set_title("Value/Brightness Channel")
        axes[0, 3].axis("off")

        # Row 2: Exclusion masks
        axes[1, 0].imshow(masks["green_exclusion"], cmap="Greens")
        axes[1, 0].set_title("Green Exclusion")
        axes[1, 0].axis("off")

        axes[1, 1].imshow(masks["light_brown"], cmap="copper")
        axes[1, 1].set_title("Light Brown Exclusion")
        axes[1, 1].axis("off")

        axes[1, 2].imshow(masks["dark_brown"], cmap="copper")
        axes[1, 2].set_title("Dark Brown Exclusion")
        axes[1, 2].axis("off")

        axes[1, 3].imshow(masks["combined_exclusion"], cmap="Reds")
        axes[1, 3].set_title("Combined Exclusion (Green+Brown)")
        axes[1, 3].axis("off")

        # Row 3: Flower type detections
        axes[2, 0].imshow(masks["colorful_flowers"], cmap="gray")
        axes[2, 0].set_title("Colorful Flowers")
        axes[2, 0].axis("off")

        axes[2, 1].imshow(masks["white_flowers"], cmap="gray")
        axes[2, 1].set_title("White Flowers")
        axes[2, 1].axis("off")

        axes[2, 2].imshow(masks["yellow_flowers"], cmap="gray")
        axes[2, 2].set_title("Yellow Flowers")
        axes[2, 2].axis("off")

        axes[2, 3].imshow(masks["pale_flowers"], cmap="gray")
        axes[2, 3].set_title("Pale/Pastel Flowers")
        axes[2, 3].axis("off")

        # Row 4: Processing steps and final result
        axes[3, 0].imshow(masks["combined_flower_mask_raw"], cmap="gray")
        axes[3, 0].set_title("Combined Raw Mask")
        axes[3, 0].axis("off")

        axes[3, 1].imshow(masks["combined_flower_mask_cleaned"], cmap="gray")
        axes[3, 1].set_title("After Morphological Cleanup")
        axes[3, 1].axis("off")

        axes[3, 2].imshow(masks["final_mask"], cmap="gray")
        axes[3, 2].set_title("Final Mask (Size Filtered)")
        axes[3, 2].axis("off")

        axes[3, 3].imshow(masks["final_result"])
        axes[3, 3].set_title(f"Final Result ({len(bboxes)} flowers)")
        axes[3, 3].axis("off")

        plt.tight_layout()
        plt.show()

    else:
        # Simple before/after comparison
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

        ax1.imshow(masks["original_image"])
        ax1.set_title("Original Image")
        ax1.axis("off")

        ax2.imshow(masks["final_result"])
        ax2.set_title(f"Flower Detection Results (n={len(bboxes)})")
        ax2.axis("off")

        plt.tight_layout()
        plt.show()

    # Print detection summary
    print(f"\\nDetection Summary:")
    print(f"Image size: {masks['detection_info']['image_size']}")
    print(f"Detected {len(bboxes)} potential flowers:")

    for i, bbox in enumerate(bboxes):
        x1, y1, x2, y2 = bbox["x1"], bbox["y1"], bbox["x2"], bbox["y2"]
        w, h = x2 - x1, y2 - y1
        flower_type = bbox.get("flower_type", "other")
        hue = bbox.get("avg_hue", 0)
        brightness = bbox.get("avg_brightness", 0)
        saturation = bbox.get("avg_saturation", 0)

        print(f"  {flower_type.title()} Flower {i + 1}:")
        print(f"    Location: [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")
        print(f"    Size: {w:.0f}x{h:.0f} px ({bbox['area']:.0f} px²)")
        print(f"    Confidence: {bbox['confidence']:.2f}")
        print(f"    HSV: H={hue:.1f}°, S={saturation:.1f}, V={brightness:.1f}")

    return bboxes, masks

In [ ]:
def generate_color_based_bboxes(image_dir, output_dir, test_mode=True, max_images=10):
    """
    Generate color-based bounding boxes for all images and save to JSON.

    Args:
        image_dir: Directory containing flower images
        output_dir: Directory to save bbox JSON files
        test_mode: If True, only process max_images for testing
        max_images: Maximum images to process in test mode
    """
    image_dir = Path(image_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)

    # Get image files
    image_extensions = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(image_dir.glob(ext))

    if test_mode and len(image_paths) > max_images:
        image_paths = image_paths[:max_images]
        print(f"Test mode: processing first {max_images} images")

    print(f"Processing {len(image_paths)} images...")

    results_summary = {
        "method": "color_based_non_green",
        "total_images": len(image_paths),
        "successful_detections": 0,
        "total_flowers_found": 0,
        "results": [],
    }

    for i, image_path in enumerate(tqdm(image_paths, desc="Processing images")):
        try:
            # Run color detection
            bboxes = detect_non_green_flowers(image_path)

            # Clean bbox format (remove extra fields for JSON)
            clean_bboxes = []
            for bbox in bboxes:
                clean_bboxes.append(
                    {
                        "x1": bbox["x1"],
                        "y1": bbox["y1"],
                        "x2": bbox["x2"],
                        "y2": bbox["y2"],
                    }
                )

            # If no bboxes found, create a centered default one
            if not clean_bboxes:
                # Load image to get dimensions
                img = cv2.imread(str(image_path))
                if img is not None:
                    h, w = img.shape[:2]
                    # Create centered bbox (1/4 of image size)
                    bbox_w, bbox_h = w // 4, h // 4
                    x1 = (w - bbox_w) // 2
                    y1 = (h - bbox_h) // 2
                    x2 = x1 + bbox_w
                    y2 = y1 + bbox_h

                    clean_bboxes = [
                        {
                            "x1": float(x1),
                            "y1": float(y1),
                            "x2": float(x2),
                            "y2": float(y2),
                        }
                    ]
                    print(
                        f"No flowers detected in {image_path.name}, using centered default"
                    )

            # Save to JSON file
            json_filename = f"{image_path.stem}.json"
            json_path = output_dir / json_filename

            bbox_data = {"bboxes": clean_bboxes}
            with open(json_path, "w") as f:
                json.dump(bbox_data, f, indent=2)

            # Update summary
            results_summary["successful_detections"] += 1
            results_summary["total_flowers_found"] += len(clean_bboxes)
            results_summary["results"].append(
                {
                    "image": image_path.name,
                    "flowers_found": len(clean_bboxes),
                    "json_file": json_filename,
                }
            )

        except Exception as e:
            print(f"Error processing {image_path.name}: {e}")
            results_summary["results"].append(
                {"image": image_path.name, "flowers_found": 0, "error": str(e)}
            )

    # Save summary
    summary_path = output_dir / "color_detection_summary.json"
    with open(summary_path, "w") as f:
        json.dump(results_summary, f, indent=2)

    print(f"\n=== Color-Based Detection Summary ==")
    print(f"Total images: {results_summary['total_images']}")
    print(f"Successful detections: {results_summary['successful_detections']}")
    print(f"Total flowers found: {results_summary['total_flowers_found']}")
    if results_summary["successful_detections"] > 0:
        avg_flowers = (
            results_summary["total_flowers_found"]
            / results_summary["successful_detections"]
        )
        print(f"Average flowers per image: {avg_flowers:.1f}")
    print(f"Results saved to: {output_dir}")
    print(f"Summary saved to: {summary_path}")

    return results_summary

In [ ]:
# Test color detection on a single image first
sample_images = list(PHOTOS.glob("*.jpg"))
bboxes, masks, img = detect_non_green_flowers(sample_images[36], return_masks=True)

plt.figure(figsize=(12, 4))
plt.subplot(131)
plt.imshow(masks["green_exclusion"], cmap="gray")
plt.subplot(132)
plt.imshow(masks["brown_exclusion"], cmap="gray")
plt.subplot(133)
plt.imshow(masks["final_result"])

# strategy for creating ds

Our main code uses a tight pad of 20, and a medium pad of 130.  
We could simply create dataset like this, it seems the easiest thing to do honestly.  

Technically, we can create simple and strandard classifiers instead of the fancy ones. lets just add a padding of 50, just use tight pad and we good.  

For generating the dataset, we need to:
- take crops from the original image 
    - I really dont want to detect ground thuogh. 
    - Actually, lets not overthink lol
- take crops from mapillary data
  - take crops from trash category data (in our actual dataset only lol)


Now do i want random resized crop? Not sure if that is a good idea. I also need to shrink the dataset lol.  
We can do something like:
- assume flower takes 10% space, (so say it takes 100 pixels, we now want it to come to 20 pxiels), shrink by 5, then take crop
- assume 20%, shrink by 10

Since i dont have bounding box, it is harder for me to understand what to use

In [ ]:
# Generate color-based bboxes for your flower images
# This will create better starting bboxes than random ones

color_bboxes_dir = INAT / "color_bboxes"

print("Generating color-based bounding boxes...")
summary = generate_color_based_bboxes(
    image_dir=PHOTOS,
    output_dir=color_bboxes_dir,
    test_mode=True,
    max_images=10,  # Adjust this number
)

In [ ]:
# Now use the widget with color-based bboxes instead of random ones!
widget = ImageReviewWidget(
    image_dir=PHOTOS,
    out_dir=SELECTED,
    skip_dir=SKIP,
    bbox_dir=color_bboxes_dir,  # Use color-based detections
)

# Note: This will show the color-detected bboxes and allow interactive editing
widget.ui()